In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

In [5]:
# old_dir = Path(r"Z:/cold_data_storage/users/roelfsemam/temp/processed/base_run")
# new_dir = Path(r"K:/PythonWork/downscaling/Kaya_downscaling/data/processed/base_run")

# scenario_run = "IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net"
# em_file = "Emissions_CO2_Excl_shipping_aviation_AFOLU_harmonised_SSP2.nc"

profile = "sensitivity_4"
old_dir = Path(f"Z:/cold_data_storage/users/roelfsemam/temp/processed/{profile}")
new_dir = Path(f"K:/PythonWork/downscaling/Kaya_downscaling/data/processed/{profile}")

scenario_run = "IMAGE_ScenarioMIP_IMAGE 3.4_Medium - SSP2_conv_year_2150_net"
em_file = "Emissions_CO2_Excl_shipping_aviation_AFOLU_harmonised_SSP2.nc"

In [6]:
xr_emissions_old = xr.open_dataset(old_dir / scenario_run / em_file)
xr_emissions_new = xr.open_dataset(new_dir / scenario_run / em_file)
xr_emissions_new

<xarray.Dataset> Size: 940MB
Dimensions:                                     (time: 12, y: 1800, x: 3600)
Coordinates:
  * x                                           (x) float64 29kB -179.9 ... 1...
  * y                                           (y) float64 14kB 89.95 ... -8...
  * time                                        (time) int64 96B 2020 ... 2100
    spatial_ref                                 int64 8B ...
    region_number                               (y, x) int8 6MB ...
Data variables:
    Emissions_CO2_Excl_shipping_aviation_AFOLU  (time, y, x) float64 622MB ...
    correction_factor                           (time, y, x) float32 311MB ...

In [7]:
#calculate global emissions per year for both datasets
global_per_year_old = xr_emissions_old["Emissions_CO2_Excl_shipping_aviation_AFOLU"].sum(dim=["y", "x"])
global_per_year_new = xr_emissions_new["Emissions_CO2_Excl_shipping_aviation_AFOLU"].sum(dim=["y", "x"])

In [8]:
#convert to pandas DataFrames
df_old = global_per_year_old.to_dataframe().reset_index()
df_new = global_per_year_new.to_dataframe().reset_index()

df_old.drop(columns=["spatial_ref"], inplace=True)
df_old["dataset"] = "old"
df_old.rename(columns={"time": "year"}, inplace=True)
df_new.drop(columns=["spatial_ref"], inplace=True)
df_new["dataset"] = "new"
df_new.rename(columns={"time": "year"}, inplace=True)

df = pd.concat([df_old, df_new])
df = df.pivot(index="year", columns="dataset", values="Emissions_CO2_Excl_shipping_aviation_AFOLU")
df["diff_abs"] = df["new"] - df["old"]
df["diff_rel"] = df["diff_abs"] / df["old"] * 100
df

dataset,new,old,diff_abs,diff_rel
year,,,,
2020,3.376895e+10,3.376895e+10,0.000000,0.000000e+00
2025,3.615148e+10,3.615148e+10,0.000000,0.000000e+00
2030,3.543192e+10,3.543192e+10,0.000000,0.000000e+00
2035,3.421346e+10,3.421346e+10,0.000000,0.000000e+00
2040,3.292861e+10,3.292861e+10,0.000000,0.000000e+00
2045,3.121916e+10,3.121916e+10,0.000000,0.000000e+00
2050,2.939413e+10,2.939413e+10,0.000000,0.000000e+00
2060,2.785149e+10,2.785149e+10,0.000000,0.000000e+00
2070,2.668293e+10,2.668293e+10,-0.000004,-1.429639e-14
